In [1]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy":     DATA_RAW / "enrichment-success-andy.json",
    "dishita":  DATA_RAW / "enrichment-success-dish.json",
    "riya":     DATA_RAW / "enrichment-success-riya.json",
    # "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]
N_MOOD_CLUSTERS = 4
TOP_N_CONTENT = 300
TOP_N_TAG = 200
TOP_N_NEIGHBOR = 150
FINAL_CANDIDATE_K = 1000
TOP_K_TAGS = 10
MIN_NEIGHBOR_MS = 30_000
MAX_NEIGHBOR_SKIP = 0.3

In [ ]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track
Tries all_tags first, falls back to sc_genres, then lastfm_tags
If none are available, returns an empty list
"""
def resolve_tags(row) -> list:
    """Canonical tag: all_tags → sc_genres → lastfm_tags."""
    for col in ("all_tags", "sc_genres", "lastfm_tags"):
        val = row.get(col)
        if isinstance(val, list) and len(val) > 0:
            return [t.strip().lower() for t in val]
    return []

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df
Filters out non-track plays (episodes, audiobooks), parses timestamps, and
adds flag for plays (>=30 seconds) that were not skipped.
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    df = pd.json_normalize(raw)
    df['user'] = user

    af_rename = {f"audio_features.{feat}": feat for feat in AUDIO_FEATURES}
    df = df.rename(columns=af_rename)

    # Keep only track plays
    df = df[df["spotify_track_uri"].notna() & df["episode_name"].isna()].copy()

    df["ts"]        = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"]   = df["skipped"].fillna(False).astype(bool)
    df["tags"]      = df.apply(resolve_tags, axis=1)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= 30_000)
    return df

"""
Loads and combines listening history for all users into a single df
Each row represents one play event, tagged with the user it belongs to
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = [load_user_events(p, u) for u, p in user_files.items()]
    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 25,868
Unique tracks: 5,923
Users: <StringArray>
['andy', 'dishita', 'riya']
Length: 3, dtype: str


In [ ]:
# SCORING

# RE-RANKING

# UI-WIDGETS (?)

# TESTING

# DEMO (?)